# ARGUS Phase 6 - notebook 03: RFT-v2 (Replay Mix) on Colab

**Hardware**: Runtime > Change runtime type > **T4 GPU**.

**Objective**: Train `argus-m1-rft-v2` using Recipe (a) from `ml/train_m1/v2/PLAN_V2.md`:
- Dataset: `v2_train.jsonl` (Replay Mix: ~50% SFT replay + ~50% diverse verified candidates).
- Hyperparameters: `lr = 2e-5`, LoRA `r = 16`, `1 epoch`.
- Evaluation: on `A_val` every 50 steps.
- Checkpointing: save adapter to Drive every 50 steps.
- Export: GGUF `q4_k_m` for Ollama registration as `argus-m1-rft-v2`.

In [ ]:
%%capture
# 1. Environment installation (Unsloth Qwen3.5 stack)
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# 2. Mount Google Drive and configure settings
from google.colab import drive
drive.mount("/content/drive")

import os, json, random, hashlib, glob

DATA_DIR   = "/content/drive/MyDrive/argus/data"      # folder with v2_train.jsonl, sft_val.jsonl, heldout_hashes.txt
OUT_DIR    = "/content/drive/MyDrive/argus/out"       # destination for adapters, logs, GGUF
BASE_MODEL = "unsloth/Qwen3.5-4B"

# If continuing from the Phase 5 SFT adapter, set the path below (leave empty string to train from base model):
SFT_ADAPTER = "/content/drive/MyDrive/argus/out/sft_adapter"  # or empty string: ""

# Hyperparameters (from PLAN_V2.md):
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 1
LORA_R        = 16
LORA_ALPHA    = 16
BATCH_SIZE    = 8
GRAD_ACCUM    = 2
EVAL_STEPS    = 50
SAVE_STEPS    = 50
MAX_LEN       = 1024

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/rft_v2", exist_ok=True)

def read_jsonl(path):
    with open(path, encoding="utf-8") as h:
        return [json.loads(l) for l in h if l.strip()]

In [ ]:
# 3. Helpers: model loader, chat formatting, tokenizer
import torch
from unsloth import FastLanguageModel

def load_model(name, max_len):
    try:
        m, t = FastLanguageModel.from_pretrained(name, max_seq_length=max_len, load_in_4bit=False, use_gradient_checkpointing="unsloth")
    except Exception as e:
        print("FastLanguageModel failed (%s); using FastVisionModel" % str(e)[:150])
        from unsloth import FastVisionModel
        m, t = FastVisionModel.from_pretrained(name, load_in_4bit=False, use_gradient_checkpointing="unsloth")
    return m, t

def add_lora(m, r=16, alpha=16):
    return FastLanguageModel.get_peft_model(m, r=r, lora_alpha=alpha, lora_dropout=0, bias="none", random_state=3407,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], use_gradient_checkpointing="unsloth")

def text_tok(t):
    return getattr(t, "tokenizer", t)

def prompt_text(tok, row):
    msgs = [{"role":"system","content":row["system"]},{"role":"user","content":row["user"]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def encode(tok, row, max_len):
    p = tok(prompt_text(tok, row), add_special_tokens=False)["input_ids"]
    c = tok(row["assistant"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = (p + c)[:max_len]
    labels = ([-100] * len(p) + c)[:max_len]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

@torch.no_grad()
def generate_greedy(model, tok, rows, max_new=300, batch=16):
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(rows), batch):
        chunk = rows[i:i+batch]
        enc = tok([prompt_text(tok, r) for r in chunk], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        new = gen[:, enc["input_ids"].shape[1]:]
        texts = tok.batch_decode(new, skip_special_tokens=True)
        for t in texts:
            outs.append(t.strip())
    return outs

In [ ]:
# 4. Load datasets & verify leakage guards
train_rows = read_jsonl(f"{DATA_DIR}/v2_train.jsonl")
print(f"Loaded {len(train_rows)} training rows from v2_train.jsonl")

# Leakage guard check:
held_path = f"{DATA_DIR}/heldout_hashes.txt"
if os.path.exists(held_path):
    held = set(open(held_path).read().split())
    bad = [r["id"] for r in train_rows if r.get("full_hash") in held]
    assert not bad, f"FATAL LEAKAGE: {len(bad)} training examples found in held-out list!"
    print("Leakage guard passed: 0 held-out hashes in training rows.")

# Validation dataset (A_val):
val_path = f"{DATA_DIR}/sft_val.jsonl" if os.path.exists(f"{DATA_DIR}/sft_val.jsonl") else f"{DATA_DIR}/val_distinct.jsonl"
val_rows = read_jsonl(val_path)
print(f"Loaded {len(val_rows)} validation rows from {val_path}")

In [ ]:
# 5. Train RFT-v2 with evaluation and checkpoint saving every 50 steps
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig

init_model_path = SFT_ADAPTER if (SFT_ADAPTER and os.path.exists(SFT_ADAPTER)) else BASE_MODEL
print("Loading base model from:", init_model_path)
model, tokenizer = load_model(init_model_path, MAX_LEN)
tok = text_tok(tokenizer)

if not hasattr(model, "peft_config"):
    model = add_lora(model, r=LORA_R, alpha=LORA_ALPHA)

ds_train = Dataset.from_list([encode(tok, r, MAX_LEN) for r in train_rows])
ds_val   = Dataset.from_list([encode(tok, r, MAX_LEN) for r in val_rows])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
    args=SFTConfig(
        output_dir=f"{OUT_DIR}/rft_v2",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=None,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_LEN
    )
)

print("Starting RFT-v2 Training...")
train_stats = trainer.train()
print("Training Complete!", train_stats)

# Save final adapter
final_adapter_dir = f"{OUT_DIR}/rft_v2_final_adapter"
model.save_pretrained(final_adapter_dir)
tok.save_pretrained(final_adapter_dir)
print(f"Saved final adapter to {final_adapter_dir}")

In [ ]:
# 6. Generate validation predictions for each 50-step checkpoint
# This allows evaluating checkpoints locally using `python -m ml.train_m1.v2.eval_v2`
import gc
val_distinct = read_jsonl(f"{DATA_DIR}/val_distinct.jsonl") if os.path.exists(f"{DATA_DIR}/val_distinct.jsonl") else val_rows

ckpts = sorted(glob.glob(f"{OUT_DIR}/rft_v2/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
print(f"Found {len(ckpts)} checkpoints:", ckpts)

try: del model, trainer
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

for ck in ckpts:
    step_num = ck.rsplit("-", 1)[1]
    pred_file = f"{OUT_DIR}/rft_v2/preds_step_{step_num}.jsonl"
    print(f"Evaluating checkpoint step {step_num}...")
    m, t = load_model(ck, 2048)
    FastLanguageModel.for_inference(m)
    tk = text_tok(t)
    outs = generate_greedy(m, tk, val_distinct, batch=16)
    with open(pred_file, "w", encoding="utf-8") as h:
        for r, o in zip(val_distinct, outs):
            h.write(json.dumps({"input_key": r["input_key"], "content": o}) + "\n")
    print(f"Wrote {pred_file}")
    del m, t; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 7. Export the best checkpoint to GGUF (Q4_K_M)
# Set CHOSEN_CKPT to the best checkpoint folder (e.g. checkpoint with best A_val)
CHOSEN_CKPT = f"{OUT_DIR}/rft_v2_final_adapter"  # or e.g. f"{OUT_DIR}/rft_v2/checkpoint-50"

print(f"Exporting {CHOSEN_CKPT} to GGUF Q4_K_M...")
m, t = load_model(CHOSEN_CKPT, 2048)
GGUF_DIR = f"{OUT_DIR}/argus_rft_v2_gguf"
os.makedirs(GGUF_DIR, exist_ok=True)
m.save_pretrained_gguf(GGUF_DIR, t, quantization_method="q4_k_m")
print("GGUF export finished:", os.listdir(GGUF_DIR))
print("NEXT: Download the .gguf file to your laptop, create the Modelfile, and register in Ollama as argus-m1-rft-v2.")